# 1. Import Libraries


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import numpy as np
from datetime import datetime

# 2. Constants


In [ ]:
url = 'https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29'
table_attribs = ['Country', 'GDP_USD_millions']
db_name = 'World_Economies.db'
table_name = 'Countries_by_GDP'
csv_path = '/home/project/Countries_by_GDP.csv'

# 3. Extract Data


In [ ]:
# Loading the entire webpage for Webscraping
html_page = requests.get(url).text

In [ ]:
html_page

In [ ]:
# Parse the text in the HTML format using BeautifulSoup
data = BeautifulSoup(html_page, 'html.parser')

In [ ]:
data

In [ ]:
# Loop to extract data in table (tbody)
tables = data.find_all('tbody')

In [ ]:
tables

In [ ]:
rows = tables[2].find_all('tr')

In [ ]:
rows

In [ ]:
df = pd.DataFrame(columns=table_attribs)
for row in rows:
        col = row.find_all('td')
        if len(col)!=0:
            if col[0].find('a') is not None and '—' not in col[2]:
                data_dict = {table_attribs[0]: col[0].a.contents[0],
                             table_attribs[1]: col[2].contents[0]}
                df1 = pd.DataFrame(data_dict, index=[0])
                df = pd.concat([df,df1], ignore_index=True)

In [ ]:
df

### Final extracting data function


In [ ]:
def extract(url, table_attribs):
	''' This function extracts the required
	information from the website and saves it to a dataframe. The
	function returns the dataframe for further processing. '''

	# Loading the entire webpage for Webscraping
	html_page = requests.get(url).text

	# Parse the text in the HTML format using BeautifulSoup
	data = BeautifulSoup(html_page, 'html.parser')

	# Loop to extract data in table (tbody)
	tables = data.find_all('tbody')

	# Select 3rd table
	rows = tables[2].find_all('tr')

	# Create dataframe with table_attribs columns
	df = pd.DataFrame(columns=table_attribs)

	for row in rows:
			col = row.find_all('td')
			if len(col)!=0:
				if col[0].find('a') is not None and '—' not in col[2]:
					data_dict = {table_attribs[0]: col[0].a.contents[0],
								table_attribs[1]: col[2].contents[0]}
					df1 = pd.DataFrame(data_dict, index=[0])
					df = pd.concat([df,df1], ignore_index=True)
		
	return df

In [ ]:
df = extract(url, table_attribs)

In [ ]:
df

# 4. Transform Data


In [ ]:
df.info()

In [ ]:
transformed_df = df.copy(deep=True)

In [ ]:
# Remove ","
transformed_df['GDP_USD_millions'] = transformed_df['GDP_USD_millions'].astype(str).str.replace(",", "")
transformed_df

In [ ]:
# Change the data type of GDP_USD_millions from object to numeric
# 1. Using astype(float) This is the simplest method when you are sure all values are numeric strings.
# 2. Using pd.to_numeric() This method is safer for messy data. With errors='coerce', non-convertible values become NaN.
transformed_df['GDP_USD_millions'] = pd.to_numeric(transformed_df['GDP_USD_millions'], errors="coerce")
print(f"Data type after changed format in GDP_USD_millions: {transformed_df['GDP_USD_millions'].dtypes}")

In [ ]:
# Change GDP_USD_millions to GDP_USD_billions by dividing ll these values by 1000 and round it to 2 decimal places.
transformed_df['GDP_USD_millions'] = (transformed_df['GDP_USD_millions']/1000).round(2)
transformed_df

In [ ]:
# Rename
transformed_df.rename(columns={"GDP_USD_millions":"GDP_USD_billions"}, inplace=True)
transformed_df